In [ ]:
import os, importlib

REPO_URL = 'https://github.com/litcorp0/checkmaize.git'  # change if you fork the project

def repo_root():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            os.system(f'cd {repo} && git clean -fdq data/manifests')
            os.system(f'cd {repo} && git pull')
            return repo
        print('Repo not on this runtime yet. Cloning from GitHub...')
        result = os.system(f'git clone {REPO_URL} /content/checkmaize')
        if result != 0 or not os.path.exists(repo):
            print('Automatic clone failed. Likely causes:')
            print('  - the GitHub repo is private (make it public first), or')
            print('  - no internet on this runtime.')
            print('Manual fix - run this in a NEW cell, then re-run this cell:')
            print(f'  !git clone {REPO_URL} /content/checkmaize')
            print('Or drag the checkmaize folder into the Colab file explorer (into /content).')
            raise SystemExit
        return repo
    return os.path.abspath('..')

REPO = repo_root()
os.chdir(REPO)
print('Working in:', os.getcwd())

missing = []
for mod in ['numpy', 'PIL', 'pandas', 'yaml', 'sklearn', 'matplotlib', 'onnx', 'onnxruntime', 'pytest']:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
if missing:
    print('installing missing packages:', missing)
    !pip install -q -r requirements.txt
    print('dependencies installed')
else:
    print('dependencies OK')

try:
    import torch
    print('torch:', torch.__version__, '| GPU available:', torch.cuda.is_available())
except ImportError:
    print('torch is not installed in this environment. Install it (GPU build) before continuing.')


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
winner = 'efficientnet_b0'
assert os.path.exists(f'artifacts/runs/{winner}/best.pt'), f'no checkpoint for {winner}'
print('winner:', winner)


In [ ]:
import os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
try:
    winner
except NameError:
    winner = 'efficientnet_b0'  # SET THIS to the model chosen from Notebook 02's scoreboard

!python -m inference.export --checkpoint artifacts/runs/{winner}/best.pt --out artifacts/runs/{winner}/model.onnx
!python -m inference.quantize --fp32 artifacts/runs/{winner}/model.onnx --out artifacts/model_int8.onnx
!python -m inference.verify --checkpoint artifacts/runs/{winner}/best.pt --fp32 artifacts/runs/{winner}/model.onnx --int8 artifacts/model_int8.onnx


In [ ]:
import csv, json, os
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
try:
    winner
except NameError:
    winner = 'efficientnet_b0'  # SET THIS to the model chosen from Notebook 02's scoreboard

labels = ['common_rust', 'gray_leaf_spot', 'northern_leaf_blight', 'healthy']
with open('artifacts/labels.json', 'w') as f:
    json.dump(labels, f)
with open('artifacts/runs/' + winner + '/metrics.json') as f:
    m = json.load(f)
with open('benchmarks/report/comparison.csv') as f:
    row = [r for r in csv.DictReader(f) if r['model'] == winner][0]
with open('inference/verify_report.json') as f:
    v = json.load(f)
metrics = {
    'model': winner,
    'test_accuracy': m['accuracy'],
    'macro_f1': m['macro_f1'],
    'onnx_bytes': int(row['onnx_bytes']),
    'int8_test_accuracy': v['int8_accuracy'],
    'shipped': 'int8' if v['ship_int8'] else 'fp32',
}
with open('artifacts/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


In [ ]:
import os, shutil
def _repo():
    if os.path.exists('/content'):
        repo = '/content/checkmaize'
        if os.path.exists(repo):
            return repo
        raise SystemExit('Run Cell 1 first (it clones the repo and installs missing packages).')
    return os.path.abspath('..')
os.chdir(_repo())
shutil.make_archive('/content/artifacts', 'zip', 'artifacts')
shutil.make_archive('/content/fixtures', 'zip', 'app/src/ml/__tests__/fixtures')
try:
    from google.colab import files
    files.download('/content/artifacts.zip')
    files.download('/content/fixtures.zip')
    files.download('docs/onnx-contract.md')
    print('downloads started (browser Colab)')
except Exception:
    print('VS Code mode: files are at:')
    print('  /content/artifacts.zip        -> unzip, copy the 3 files into checkmaize/app/assets/model/')
    print('  /content/fixtures.zip         -> unzip, copy the 2 files into checkmaize/app/src/ml/__tests__/fixtures/ (replace)')
    print('  docs/onnx-contract.md (in the repo) -> it is already in place')
    print('Drag the zips from the file explorer onto your computer.')
